Downloading Datasets:

In [ ]:
%pip install requests numpy pandas
import requests, numpy as np, pandas as pd


Note: you may need to restart the kernel to use updated packages.


In [57]:
###get biking data 
bridge_url = 'https://data.seattle.gov/api/v3/views/65db-xm6k/query.json'
r = requests.get(bridge_url).json()

In [58]:
###initially process data 
###drop irrelevant information, convert time to datetime object
r = pd.DataFrame(r).drop(columns=[':id',':version',':updated_at',':created_at'])
r.date = pd.to_datetime(r.date, format='%Y-%m-%dT%H:%M:%S.%f')

#convert strings to floats 
num_cols = r.select_dtypes(str).columns.to_list()
r[num_cols] = r[num_cols].apply(lambda x: pd.to_numeric(x), axis=0)

In [59]:
print('10 highest Days:\n_____________________')
r.sort_values('fremont_bridge', ascending=False)[:10]

10 highest Days:
_____________________


,date,fremont_bridge,fremont_bridge_nb,fremont_bridge_sb
63318,2019-06-11 17:00:00,1097.0,850.0,247.0
61470,2019-08-27 17:00:00,1071.0,826.0,245.0
62982,2019-06-25 17:00:00,1052.0,795.0,257.0
63798,2019-05-22 17:00:00,1004.0,745.0,259.0
61806,2019-08-13 17:00:00,998.0,780.0,218.0
61278,2019-09-04 17:00:00,997.0,790.0,207.0
63486,2019-06-04 17:00:00,995.0,742.0,253.0
63342,2019-06-10 17:00:00,993.0,765.0,228.0
61782,2019-08-14 17:00:00,989.0,737.0,252.0
61110,2019-09-11 17:00:00,983.0,768.0,215.0


In [82]:
####save data to csv 
r.to_csv('scraped_data/fremont_bridge_counter.csv')

bike_data = r 

In [179]:
###set up weather data api 

import os, requests, pandas as pd, time, json 

API_key = os.getenv('weather-api')
lat, lon = [47.61191234063561, -122.33851982527909]
units = 'imperial'

####get historical weather data for period the fremont bridge counter is active 
from datetime import datetime
###set start date to unix version of fremont bridge csv
start = bike_data.date.min().timestamp().__int__()
end = bike_data.date.max().timestamp().__int__()

###set up url with end as current time - will move end back according to next api call, etc 
weather_url = f'https://api.openweathermap.org/data/4.0/onecall/timeline/1h?lat={lat}&lon={lon}&units={units}&start={start}&appid={API_key}'

datetime.fromtimestamp(start), 'we need to make about 7000 calls for the data we need.'

(datetime.datetime(2012, 10, 2, 6, 0),
 'we need to make about 7000 calls for the data we need.')

In [274]:
###loop to get data from start time to end time, not the most efficient but it works for now 
weather_data = pd.DataFrame()
CHECKPOINT = 'weather_checkpoint.pk1'

###check if there is a more recent checkpoint to start off from 
if os.path.exists(CHECKPOINT):
    records = pd.read_pickle(CHECKPOINT).to_dict('records')
    start = max(rec['dt'] for rec in records) + 1 
    print(f'Resuming from {datetime.fromtimestamp(start)} - {len(records)} hours of data already saved.')
else:
    records = []
    start = bike_data.date.min().timestamp()

###initialization
loop_url = weather_url
calls = 0
session = requests.Session()

try:
    while weather_url and start < end:

        print('Data is within range. Requesting...')
        resp = requests.get(loop_url, timeout=30)
        resp.raise_for_status()
        r = resp.json()

    ##if no response, break 
        batch = r.get('data', [])
        if not batch: 
            break

        records.extend(batch)
        calls += 1

        ###ends when date end is reached 
        if batch[-1]['dt'] >= end:
            break 

        ###save every 20 calls just in case 
        if calls % 20:
            pd.DataFrame(records).to_pickle('scraped_data/weather_checkpoint.pk1')
        data_time = datetime.fromtimestamp(r['data'][0]['dt'])

        ###break at 700 calls 
        if calls == 700:
            break

        print(f'Data Saved for {data_time} - Call: {calls}')

        loop_url = r.get('next')
        time.sleep(1)

finally:
    pd.DataFrame(records).to_pickle(CHECKPOINT)
    print(f'{len(records)} hourly records saved. ')

    

Resuming from 2012-10-06 09:00:01 - 100 hours of data already saved.
Data is within range. Requesting...
Data Saved for 2012-10-02 06:00:00 - Call: 1
Data is within range. Requesting...
Data Saved for 2012-10-03 02:00:00 - Call: 2
Data is within range. Requesting...
Data Saved for 2012-10-03 22:00:00 - Call: 3
Data is within range. Requesting...
Data Saved for 2012-10-04 18:00:00 - Call: 4
Data is within range. Requesting...
Data Saved for 2012-10-05 14:00:00 - Call: 5
Data is within range. Requesting...
Data Saved for 2012-10-06 10:00:00 - Call: 6
Data is within range. Requesting...
Data Saved for 2012-10-07 06:00:00 - Call: 7
Data is within range. Requesting...
Data Saved for 2012-10-08 02:00:00 - Call: 8
Data is within range. Requesting...
Data Saved for 2012-10-08 22:00:00 - Call: 9
Data is within range. Requesting...
Data Saved for 2012-10-09 18:00:00 - Call: 10
Data is within range. Requesting...
Data Saved for 2012-10-10 14:00:00 - Call: 11
320 hourly records saved. 


KeyboardInterrupt: 

In [275]:
weather_df = pd.DataFrame(records).drop_duplicates('dt').sort_values('dt')
weather_df = weather_df[weather_df['dt'] <= end]
weather_df['date'] = (pd.to_datetime(weather_df['dt'], unit='s', utc=True).dt.tz_convert('America/Los_Angeles'))
weather_df.to_csv('scraped_data/weather.csv')

In [276]:
weather_df

,dt,temp,feels_like,pressure,humidity,dew_point,uvi,clouds,visibility,wind_speed,wind_deg,wind_gust,weather,date
0,1349182800,52.93,51.35,1022,73,44.53,0,31,10000,11.50,180,18.39,"[{'id': 802, 'main': 'Clouds', 'description': ...",2012-10-02 06:00:00-07:00
1,1349186400,53.04,51.71,1022,78,46.38,0,29,10000,10.36,180,12.64,"[{'id': 802, 'main': 'Clouds', 'description': ...",2012-10-02 07:00:00-07:00
2,1349190000,54.52,52.63,1022,63,42.21,0,30,10000,11.50,180,19.55,"[{'id': 802, 'main': 'Clouds', 'description': ...",2012-10-02 08:00:00-07:00
3,1349193600,55.63,53.44,1022,54,39.27,0,29,10000,6.89,180,10.36,"[{'id': 802, 'main': 'Clouds', 'description': ...",2012-10-02 09:00:00-07:00
4,1349197200,57.25,55.26,1022,55,41.25,0,43,10000,4.61,180,8.05,"[{'id': 802, 'main': 'Clouds', 'description': ...",2012-10-02 10:00:00-07:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,1349956800,49.21,49.21,1015,79,42.98,0,86,10000,1.14,225,5.75,"[{'id': 804, 'main': 'Clouds', 'description': ...",2012-10-11 05:00:00-07:00
316,1349960400,49.05,49.05,1014,80,43.16,0,80,10000,1.14,225,3.44,"[{'id': 803, 'main': 'Clouds', 'description': ...",2012-10-11 06:00:00-07:00
317,1349964000,48.69,48.69,1014,79,42.48,0,75,10000,2.30,225,6.89,"[{'id': 803, 'main': 'Clouds', 'description': ...",2012-10-11 07:00:00-07:00
318,1349967600,49.24,49.24,1015,79,43.02,0,83,10000,2.30,225,2.30,"[{'id': 803, 'main': 'Clouds', 'description': ...",2012-10-11 08:00:00-07:00
